# Проверка гипотезы и прогноз нагрузки по обращениям

Этот notebook помогает выполнить самостоятельную работу: проверить гипотезу о различии времени решения обращений между каналами `email` и `chat`, построить временной ряд обращений, сделать baseline-прогноз и оформить аналитический вывод.

Работайте сверху вниз. После каждого крупного блока проверяйте результат.

## 1. Рабочая ситуация

Вы работаете аналитиком в команде поддержки клиентов. Нужно ответить на два вопроса:

1. Отличается ли среднее время решения обращений между каналами `email` и `chat`?
2. Какой может быть ожидаемая недельная нагрузка на поддержку по прошлой динамике обращений?

## 2. Проверка окружения

Этот блок нужен для диагностики рабочей папки, версии Python и доступности библиотек. Он не решает аналитическую задачу, а помогает быстро найти технические проблемы.

In [ ]:
from pathlib import Path
import sys

print("Python:", sys.version)
print("Рабочая папка:", Path.cwd())

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy import stats
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    print("pandas:", pd.__version__)
    print("numpy:", np.__version__)
except ImportError as error:
    print("Не удалось импортировать библиотеку:", error)
    print("Установите зависимости из requirements.txt или используйте Google Colab.")

## 3. Проверка файлов

Файл с данными должен лежать по относительному пути `data/raw/support_tickets.csv`. Не используйте абсолютные пути вида `C:\Users\...`, чтобы notebook запускался на разных компьютерах.

In [ ]:
current = Path.cwd()
relative_data_path = Path("data") / "raw" / "support_tickets.csv"

# Notebook можно запускать из корня проекта или из папки notebooks.
# Сначала ищем файл от текущей папки, затем на уровень выше.
if (current / relative_data_path).exists():
    PROJECT_ROOT = current
elif (current.parent / relative_data_path).exists():
    PROJECT_ROOT = current.parent
else:
    PROJECT_ROOT = current

DATA_PATH = PROJECT_ROOT / relative_data_path
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Корень проекта:", PROJECT_ROOT)
print("Проверяем файл:")
if DATA_PATH.exists():
    print("OK:", DATA_PATH)
else:
    print("НЕ НАЙДЕН:", DATA_PATH)
    print("Проверьте, что файл support_tickets.csv лежит в data/raw внутри папки проекта.")

## 4. Загрузка данных

Сначала загрузим CSV и посмотрим первые строки. После загрузки нельзя сразу делать выводы: нужно проверить структуру и качество данных.

In [ ]:
tickets = pd.read_csv(DATA_PATH)

print("Размер таблицы:", tickets.shape)
display(tickets.head())

## 5. Первичная проверка качества

Проверим типы данных, пропуски, дубликаты и распределение категорий. Это важно перед статистическим тестом и временным рядом.

In [ ]:
print("Типы данных:")
display(tickets.dtypes)

print("Пропуски по столбцам:")
display(tickets.isna().sum())

print("Дубликаты ticket_id:", tickets["ticket_id"].duplicated().sum())

print("Каналы обращений:")
display(tickets["channel"].value_counts(dropna=False))

## 6. Подготовка данных

В этом блоке приводим дату к типу datetime, нормализуем текстовые поля и создаём очищенную таблицу для анализа.

Контрольная логика: время решения не должно быть отрицательным, дата должна быть распознана, канал должен быть в едином регистре.

In [ ]:
tickets_clean = tickets.copy()

# Приводим дату. Некорректные даты станут NaT.
tickets_clean["created_at"] = pd.to_datetime(tickets_clean["created_at"], errors="coerce")

# Нормализуем текстовые поля.
for col in ["channel", "priority", "category", "customer_segment", "region"]:
    tickets_clean[col] = tickets_clean[col].astype(str).str.strip()

tickets_clean["channel"] = tickets_clean["channel"].str.lower()
tickets_clean["category"] = tickets_clean["category"].str.lower()

print("Некорректные даты:", tickets_clean["created_at"].isna().sum())
print("Некорректные resolution_hours <= 0:", (tickets_clean["resolution_hours"] <= 0).sum())
print("Дубликаты ticket_id:", tickets_clean["ticket_id"].duplicated().sum())

In [ ]:
# Убираем дубликаты ticket_id и явно некорректные значения для анализа времени решения.
tickets_clean = tickets_clean.drop_duplicates(subset=["ticket_id"]).copy()

analysis_data = tickets_clean[
    tickets_clean["created_at"].notna()
    & tickets_clean["resolution_hours"].notna()
    & (tickets_clean["resolution_hours"] > 0)
].copy()

print("Строк в исходной таблице:", len(tickets))
print("Строк после подготовки для анализа:", len(analysis_data))

print("Каналы после очистки:")
display(analysis_data["channel"].value_counts())

## 7. Формулировка гипотезы

Запишите гипотезы перед просмотром результата теста.

**H0:** среднее время решения обращений в каналах `email` и `chat` не отличается.

**H1:** среднее время решения обращений в каналах `email` и `chat` отличается.

## 8. Описательное сравнение групп

Перед тестом посмотрим размер групп, среднее, медиану и разброс. Если группы маленькие или сильно различаются по составу, вывод нужно делать осторожно.

In [ ]:
compare_channels = ["email", "chat"]
group_data = analysis_data[analysis_data["channel"].isin(compare_channels)].copy()

channel_summary = (
    group_data
    .groupby("channel", as_index=False)
    .agg(
        tickets_count=("ticket_id", "nunique"),
        mean_resolution=("resolution_hours", "mean"),
        median_resolution=("resolution_hours", "median"),
        std_resolution=("resolution_hours", "std"),
        min_resolution=("resolution_hours", "min"),
        max_resolution=("resolution_hours", "max")
    )
)

display(channel_summary)

## 9. Статистический тест

Сравним две независимые группы через t-test. Используем `equal_var=False`, чтобы не требовать равенства дисперсий групп.

In [ ]:
email_time = group_data.loc[group_data["channel"] == "email", "resolution_hours"].dropna()
chat_time = group_data.loc[group_data["channel"] == "chat", "resolution_hours"].dropna()

alpha = 0.05
test_result = stats.ttest_ind(email_time, chat_time, equal_var=False)

print("Количество email:", len(email_time))
print("Количество chat:", len(chat_time))
print("t-statistic:", test_result.statistic)
print("p-value:", test_result.pvalue)

if test_result.pvalue < alpha:
    print("Решение: есть основание отвергнуть H0 при alpha = 0.05.")
else:
    print("Решение: недостаточно оснований отвергнуть H0 при alpha = 0.05.")

## 10. Интерпретация результата теста

Заполните вывод своими словами. Не используйте формулировку «гипотеза доказана». Лучше писать: «есть основание отвергнуть H0» или «недостаточно оснований отвергнуть H0».

**Шаблон:**

На данных за выбранный период была проверена гипотеза о различии среднего времени решения обращений между каналами `email` и `chat`. При уровне значимости 0.05 p-value составило ... . Это даёт / не даёт основание отвергнуть H0. Вывод следует использовать осторожно, потому что ... .

## 11. Подготовка временного ряда

Теперь построим недельную динамику количества обращений. Для временного ряда важен порядок дат. Сначала сортируем данные по времени, затем делаем ресемплинг по неделям.

In [ ]:
time_data = analysis_data.sort_values("created_at").copy()

weekly_tickets = (
    time_data
    .set_index("created_at")
    .resample("W")
    .agg(tickets_count=("ticket_id", "nunique"))
)

display(weekly_tickets.head())
print("Количество недель:", len(weekly_tickets))

## 12. Визуализация динамики и скользящее среднее

Скользящее среднее помогает сгладить случайные колебания и увидеть общую тенденцию. Оно не является доказательством будущего поведения, но полезно для первичного чтения динамики.

In [ ]:
weekly_tickets["moving_avg_4"] = weekly_tickets["tickets_count"].rolling(window=4).mean()

plt.figure(figsize=(10, 5))
plt.plot(weekly_tickets.index, weekly_tickets["tickets_count"], marker="o", label="Факт")
plt.plot(weekly_tickets.index, weekly_tickets["moving_avg_4"], marker="o", label="Скользящее среднее 4 недели")
plt.title("Недельная динамика обращений")
plt.xlabel("Неделя")
plt.ylabel("Количество обращений")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 13. Baseline-прогноз

Для временных рядов нельзя случайно перемешивать данные. Обучающая часть должна быть раньше тестовой. Построим простой прогноз: среднее значение последних 4 недель обучающей части.

In [ ]:
series = weekly_tickets["tickets_count"].dropna()

train = series.iloc[:-4]
test = series.iloc[-4:]

baseline_forecast = pd.Series(
    train.tail(4).mean(),
    index=test.index,
    name="forecast"
)

comparison = pd.DataFrame({
    "fact": test,
    "forecast": baseline_forecast
})

display(comparison)

## 14. Оценка ошибки прогноза

MAE показывает среднюю абсолютную ошибку в единицах показателя. RMSE сильнее штрафует крупные ошибки.

In [ ]:
mae = mean_absolute_error(comparison["fact"], comparison["forecast"])
rmse = np.sqrt(mean_squared_error(comparison["fact"], comparison["forecast"]))

print("MAE:", mae)
print("RMSE:", rmse)

comparison.to_csv(OUTPUT_DIR / "forecast_comparison.csv")
print("Файл сохранён:", OUTPUT_DIR / "forecast_comparison.csv")

## 15. График факта и прогноза

Сравним последние 4 фактические недели с baseline-прогнозом.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(comparison.index, comparison["fact"], marker="o", label="Факт")
plt.plot(comparison.index, comparison["forecast"], marker="o", label="Baseline-прогноз")
plt.title("Сравнение факта и baseline-прогноза")
plt.xlabel("Неделя")
plt.ylabel("Количество обращений")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 16. Итоговый аналитический вывод

Заполните итоговый вывод по структуре:

1. Что анализировалось.
2. Какие данные использовались.
3. Какая гипотеза проверялась.
4. Какой тест был выбран и почему.
5. Какой p-value получен.
6. Какое решение принято по H0.
7. Что видно по временной динамике.
8. Какой baseline-прогноз построен.
9. Какая ошибка прогноза получилась.
10. Какие ограничения есть у вывода.

**Место для вывода:**

...

## 17. Чек-лист завершения

- Notebook запущен сверху вниз.
- Используются относительные пути.
- Дата преобразована в datetime.
- Проверены пропуски, дубликаты и некорректные значения.
- H0 и H1 записаны явно.
- p-value сравнивается с alpha = 0.05.
- Временной ряд построен по неделям.
- Train/test split выполнен по времени.
- Forecast сравнен с fact.
- MAE и RMSE посчитаны.
- Итоговый вывод содержит ограничение.